# Thesis - Intelligent Reflecting Surface

## Dataset Generation

This section creates a synthetic dataset for Intelligent Reflecting Surface (IRS) learning.

For each user position $\mathbf{p}=[x,y]^T$, the simplified cascaded channel follows the notation used in the thesis:
$$
h_{\mathrm{eff}}(\mathbf{p},\boldsymbol{\theta})=\mathbf{h}_{\mathrm{IU}}^{H}(\mathbf{p})\,\mathrm{diag}(e^{j\boldsymbol{\theta}})\,\mathbf{h}_{\mathrm{BI}}
$$
where $\boldsymbol{\theta}=[\theta_1,\dots,\theta_M]^T$ is the IRS phase vector.
The element-wise phase-alignment rule is:
$$
\theta_m^{\star}(\mathbf{p})=-\angle\!\left(h_{\mathrm{IU},m}^{*}(\mathbf{p})\,h_{\mathrm{BI},m}\right),\quad m=1,\dots,M
$$
With $K$ quantized levels, phases are projected onto:
$$
\theta_m \in \left\{0,\frac{2\pi}{K},\dots,\frac{2\pi(K-1)}{K}\right\}
$$
Thus each training sample is a pair $(\mathbf{p}^{(i)},\boldsymbol{\theta}^{\star(i)})$.

In [7]:
import numpy as np

In [8]:
def dataset(samples=1000, size=8, K=4):
    """
    Generate synthetic UE positions and brute-force IRS phase labels.

    Args:
        samples: Number of dataset samples (UE positions).
        size: IRS side length. Total IRS elements are M = size**2.
        K: Number of quantized phase levels used in the search grid.

    Returns:
        positions: Array with shape (samples, 2) containing UE coordinates (x, y).
        theta_opt: Array with shape (samples, M) containing the best phase profile found
            for each sample according to the simplified received-power metric.
    """
    # IRS element positions on a 2D grid, with fixed height z = 1.5
    xx, yy = np.meshgrid(np.linspace(8.5, 9.5, size), np.linspace(3.5, 4.5, size), indexing="ij")
    irs_pos = np.column_stack((xx.ravel(), yy.ravel(), np.full(xx.size, 1.5)))
    # Random UE positions in the area: x in [1, 8], y in [1, 7]
    positions = np.hstack((np.random.uniform(1, 8, (samples, 1)), np.random.uniform(1, 7, (samples, 1))))
    # IRS total number of reflecting elements
    M = size**2

    phase_levels = np.linspace(0, 2 * np.pi, K, endpoint=False)
    theta_opt = np.zeros((samples, M))
    for i in range(samples):
        p = np.array([positions[i, 0], positions[i, 1], 1.0]) # UE with z = 1

        # Simplified channels: pathloss approximation + random phase
        h_BI = 10**(-3.5 / 10) * np.exp(1j * np.random.uniform(0,2 * np.pi, M)) # BS-IRS, 3.5 dB
        h_IU = np.exp(-np.linalg.norm(irs_pos - p, axis=1) / 10) * np.exp(1j * np.random.uniform(0, 2 * np.pi, M))  # IRS-UE

        g = h_IU.conj() * h_BI # channel product
        theta = -np.angle(g) # optimal continuous phase
        idx = np.argmin(np.abs(theta[:,None] - phase_levels), axis=1) # quantization (K phase levels)

        theta_opt[i] = phase_levels[idx]

    return positions, theta_opt

## Algorithm

Helper functions used across all algorithms.

The train/test split is defined as
$$
\mathcal{D}_{\mathrm{train}}=\{(\mathbf{x}^{(i)},\boldsymbol{\theta}^{\star(i)})\}_{i=1}^{N_{\mathrm{tr}}},\quad
\mathcal{D}_{\mathrm{test}}=\{(\mathbf{x}^{(i)},\boldsymbol{\theta}^{\star(i)})\}_{i=N_{\mathrm{tr}}+1}^{N},\quad N_{\mathrm{tr}}=\lfloor rN\rfloor
$$
with split ratio $r\in(0,1)$.
The error metric follows the thesis notation:
$$
\mathrm{NMSE}=\frac{\sum_{i=1}^{N}\lVert \boldsymbol{\theta}^{\star(i)}-\hat{\boldsymbol{\theta}}^{(i)} \rVert_2^2}{\sum_{i=1}^{N}\lVert \boldsymbol{\theta}^{\star(i)} \rVert_2^2},
$$
where the common factor $1/N$ in numerator and denominator is canceled.

In [ ]:
def train_test_split(x, y, train_ratio=0.8):
    split = int(train_ratio * len(x))
    return x[:split], x[split:], y[:split], y[split:] # xtrain, xtest, ytrain, ytest

def nmse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2) / np.mean(y_true ** 2) # rmse / var


### Multivariate Linear Regression

Input and output are:
$$
\mathbf{x}=[x,y]^T\in\mathbb{R}^2, \qquad \hat{\boldsymbol{\theta}}\in\mathbb{R}^{M}
$$
The regression model is
$$
\hat{\boldsymbol{\theta}}=\mathbf{W}\mathbf{x}+\mathbf{b}, \qquad \mathbf{W}\in\mathbb{R}^{M\times 2},\; \mathbf{b}\in\mathbb{R}^{M}
$$
Parameters are estimated by least squares:
$$
(\hat{\mathbf{W}},\hat{\mathbf{b}})=\arg\min_{\mathbf{W},\mathbf{b}}\sum_{i=1}^{N}\left\lVert \boldsymbol{\theta}^{\star(i)}-(\mathbf{W}\mathbf{x}^{(i)}+\mathbf{b})\right\rVert_2^2
$$
Then each predicted phase is quantized to the hardware levels
$$
\theta_m \leftarrow \arg\min_{q\in\left\{0,\frac{2\pi}{K},\dots,\frac{2\pi(K-1)}{K}\right\}}\left|\hat{\theta}_m-q\right|, \quad m=1,\dots,M
$$

In [10]:
def lg(x_train, x_test, y_train, y_test):
    X = np.concatenate([x_train, np.ones((x_train.shape[0], 1))], axis=1) # Add bias term as an extra column of ones
    theta = np.linalg.pinv(X.T @ X) @ X.T @ y_train # Closed-form least-squares solution

    W, b = theta[:-1], theta[-1]
    y_pred = x_test @ W + b

    return y_pred, nmse(y_test, y_pred)

### k-Nearest Neighbors (kNN) Regression

$$
\mathcal{D}=\left\{\left(\mathbf{x}^{(i)},\boldsymbol{\theta}^{\star(i)}\right)\right\}_{i=1}^{N}.
$$
For a query point $\mathbf{x}$, the $k$ nearest indices are
$$
\mathcal{N}_k(\mathbf{x})=\arg\min_{\substack{S\subseteq\{1,\dots,N\}\\|S|=k}}\sum_{j\in S}\lVert\mathbf{x}-\mathbf{x}^{(j)}\rVert_2
$$
The multi-output estimate is a weighted average:
$$
\hat{\boldsymbol{\theta}}(\mathbf{x})=\sum_{j\in\mathcal{N}_k(\mathbf{x})}w_j\,\boldsymbol{\theta}^{\star(j)},
$$
$$
w_j=\frac{1/(\lVert\mathbf{x}-\mathbf{x}^{(j)}\rVert_2+\epsilon)}{\sum_{\ell\in\mathcal{N}_k(\mathbf{x})}1/(\lVert\mathbf{x}-\mathbf{x}^{(\ell)}\rVert_2+\epsilon)},\quad \epsilon>0
$$
Finally, $\hat{\boldsymbol{\theta}}$ is quantized to the same $K$ phase levels used by the IRS hardware.

In [11]:
def knn(x_train, x_test, y_train, y_test, k=5):
    dists = np.linalg.norm(x_test[:, None, :] - x_train[None, :, :], axis=2) # pairwise distances (N_test × N_train)
    idx = np.argsort(dists, axis=1)[:, :k] # indices of k nearest neighbors

    neighbors_d = np.take_along_axis(dists, idx, axis=1) + 1e-8 # k nearest distances
    neighbors_y = y_train[idx] # corresponding y values

    weights = 1 / neighbors_d
    weights = weights / weights.sum(axis=1, keepdims=True)

    y_pred = np.sum(neighbors_y * weights[:, :, None], axis=1) # weighted average

    return y_pred, nmse(y_test, y_pred)

### Multi-Layer Perceptron

The third approach uses a Multi-Layer Perceptron (MLP) to learn the nonlinear mapping between user position and IRS phase profile.

The input remains
$$
\mathbf{x}=[x,y]^T\in\mathbb{R}^2,
$$
while the network predicts
$$
\hat{\boldsymbol{\theta}}\in\mathbb{R}^{M}.
$$
Before training, both inputs and targets are standardized using mean and standard deviation computed on the training set only, and the same statistics are then applied to the test set.

The implemented architecture has two hidden layers with ReLU activations and a final linear output layer. Weights are initialized with a He-type scheme, which is well suited for ReLU units and helps maintain stable signal and gradient scales during optimization.

Training is performed with mini-batch gradient descent by minimizing the mean-squared error between predictions $\hat{\boldsymbol{\theta}}$ and reference labels $\boldsymbol{\theta}^{\star}$. After training, predictions are mapped back to the original scale and performance is evaluated through the NMSE metric defined in the algorithm section.def relu(x):
    return np.maximum(0, x)

In [ ]:
def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

def map_mlp(X_train,X_test,y_train,y_test,hidden=(64, 32),lr=1e-3,epochs=100,batch_size=32,seed=42,verbose=True, print_every=20):
    """Train a simple two-hidden-layer MLP for multi-output regression."""
    X_train = np.asarray(X_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)
    y_train = np.asarray(y_train, dtype=float)
    y_test = np.asarray(y_test, dtype=float)

    h1, h2 = hidden
    input_dim = X_train.shape[1]
    output_dim = y_train.shape[1]

    rng = np.random.default_rng(seed)

    # Standardize inputs and targets using only the training split.
    x_mean = X_train.mean(axis=0, keepdims=True)
    x_std = X_train.std(axis=0, keepdims=True)
    x_std[x_std == 0] = 1.0

    y_mean = y_train.mean(axis=0, keepdims=True)
    y_std = y_train.std(axis=0, keepdims=True)
    y_std[y_std == 0] = 1.0

    X_train_scaled = (X_train - x_mean) / x_std
    X_test_scaled = (X_test - x_mean) / x_std
    y_train_scaled = (y_train - y_mean) / y_std

    # He initialization works well with ReLU activations.
    W1 = rng.standard_normal((input_dim, h1)) * np.sqrt(2.0 / input_dim)
    b1 = np.zeros((1, h1))
    W2 = rng.standard_normal((h1, h2)) * np.sqrt(2.0 / h1)
    b2 = np.zeros((1, h2))
    W3 = rng.standard_normal((h2, output_dim)) * np.sqrt(1.0 / h2)
    b3 = np.zeros((1, output_dim))

    sample_count = X_train_scaled.shape[0]

    for epoch in range(epochs):
        permutation = rng.permutation(sample_count)
        epoch_loss = 0.0

        for start in range(0, sample_count, batch_size):
            stop = start + batch_size
            batch_idx = permutation[start:stop]

            X_batch = X_train_scaled[batch_idx]
            y_batch = y_train_scaled[batch_idx]

            z1 = X_batch @ W1 + b1
            a1 = relu(z1)
            z2 = a1 @ W2 + b2
            a2 = relu(z2)
            y_pred = a2 @ W3 + b3

            batch_loss = np.mean((y_pred - y_batch) ** 2)
            epoch_loss += batch_loss * X_batch.shape[0]

            dL = 2.0 * (y_pred - y_batch) / X_batch.shape[0]
            dW3 = a2.T @ dL
            db3 = np.sum(dL, axis=0, keepdims=True)

            da2 = dL @ W3.T
            dz2 = da2 * relu_deriv(z2)
            dW2 = a1.T @ dz2
            db2 = np.sum(dz2, axis=0, keepdims=True)

            da1 = dz2 @ W2.T
            dz1 = da1 * relu_deriv(z1)
            dW1 = X_batch.T @ dz1
            db1 = np.sum(dz1, axis=0, keepdims=True)

            W1 -= lr * dW1
            b1 -= lr * db1
            W2 -= lr * dW2
            b2 -= lr * db2
            W3 -= lr * dW3
            b3 -= lr * db3

        if verbose and (epoch == 0 or (epoch + 1) % print_every == 0):
            mean_epoch_loss = epoch_loss / sample_count
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {mean_epoch_loss:.6f}")

    a1_test = relu(X_test_scaled @ W1 + b1)
    a2_test = relu(a1_test @ W2 + b2)
    y_pred_scaled = a2_test @ W3 + b3
    y_pred = y_pred_scaled * y_std + y_mean

    return y_pred, nmse(y_test, y_pred)